## Thiết kế 8 metric

1. Task success rate

- Định nghịa: % test case mà kết quả cuối cùng khớp expected_result và thỏa toàn bộ constraints.
- Đo bằng: LLM-judge so khớp semantic giữa output thực tế và expected_result, kết hợp checklist rule-based cho từng constraint (mỗi constraint có thể mã hóa thành 1 assertion)
- Công thức: success_count / total_cases
- Target: >90%

2. Tool selection accuracy

- Định nghia: % case mà tập tool agent thực sự gọi khớp với expected_tools.
- Đo bằng: So khớp set: thiếu tool bắt buộc -> fail; gọi thêm tool không cần thiết (không nguy hại) -> tính là partial-fail riêng biệt (không gộp chung để tránh làm loãng tín hiệu)
- Công thức: case đúng tool set / total_case
- Target: >90%

3. Tool argument accuracy

- Định nghĩa: Trong các case gọi đúng tool % lần tham số truyền vào khớp chính xác với giá trị agent lẽ ra phải dùng.
- Đo bằng: So khớp field-by-field giữa actual_args và expected_args
- Công thức: tham số đúng / tổng tham số băt buộc qua các lần gọi tool
- Target: >95%4

4. Trajectory quality

- Định nghĩa: Chất lượng toàn bộ chuỗi hành động - đúng thứ tự bước, không thừa/thiếu, xử lý lỗi tool đúng cách, tuân thủ constraint xuyên suốt (không chỉ câu trả lời cuối cùng).
- Đo bằng: LLM-judge chấm theo rubic dựa trên checklist: có planning failure không, có tool selection/execution failure không, có reasoning failure không..
- Công thức: Điểm trung bình rubic trên toàn bộ 50 case, họă % đặt ngưỡng pass
- Target: điểm trung bình >=4.0/5.0

5. Answer correctness

- Định nghia: Nội dung câu trả lời cuối có đúng về mặt ngữ nghĩa so với expected_result không - không cần khớp từng chữ.
- Đo bằng: LLM-judge so sánh semantic; vơi case adversarial, "correctness" = có từ chối/redirect đúng cách hay không
- Công thức: case đúng nội dung / total cases 
- Target: > 90%

6. Groundness

- Định nghĩa: % số claim/số liệu trong câu trả lời có thể truy ngược về một tool_result hoặc đoạn KB cụ thể (không hallucinate) - áp dụng nguyên lý Grounding Validator ở bài 8.
- Đo bằng: Trích các claim có số liệu/fact trong output, đối chiếu với tool_result/KB context trong trajectory. 
- Công thức: claim có nguồn/tổng claim gần grounding
- Taret: 95%

7. Latency

- Định nghĩa: Thời gian xử lý từ lúc nhận input tới khi trả lời cuối cùng cho user. 
- Đo bằng: Đo P50 và P95 trên toàn bộ 50 case; tách riêng theo case_type vì tool_failure/adversarial thường cần thêm bước retry/escalate nên latency cao hơn normal
- Target: P95 < 10 (final project), theo rõi riêng P95 của nhóm tool_failure vì đây là nhóm dễ vượt ngưỡng nhất.

8. Cost per task

- Định nghĩa: Tổng chi phí (token LLM + phí gọi tool nếu có) trung bình mõi test case
- Đo bằng: Cộng token input/output từng lần gọi LLM trong trajectory + chi phí gọi tool call (nếu tool trả phí), quy đổi theo giá model đang dùng.
- Công thức: tổng cost / 50 case., theo dõi thêm cost trung bình theo từng case_type (adversarial/tool_failure thường tốn hơn do nhiều lượt retry/escalate)
- Target: Tự xác định theo budget dự án; theo dõi xu hướng tăng giảm qua ác lần cải tiến hơn là chỉ nhìn một con số tuyệt đối.



## 5 câu hỏi bắt buộc
1. Failure mode lớn nhất: agent pass tốt ở nhóm normal nhưng sụp ở adversarial/tool_failure - vì đây là 2 nhóm dễ bị bỏ sot khi chỉ test happy path; một agent có task success rate cao chỉ trên normal case là một con số gây hiểu lầm nguy hiểm

2. Biết hệ thống tốt lên khi nào: cả 8 metric cải thiện đồng thời theo từng case_type riêng biệt (không chỉ trung bình chung) - đặt biệt groundness và tool selection accuracy trên nhóm adversarial/tool_failure,vì đây là nơi thạt sự nguy hiểm ẩn náu.

3. Metric nào cần theo dõi: ưu tiên theo dõi groundedness và tool argument accuracy trước nhất (sai ở đây là hậu quả trực tiếp gây ra cho khách hoàn), task success rate và trajectory quality theo dõi tổng quan, latency/cost theo dõi để cân bằng trade-off.

4. Có thục sự cần Agent không? Với phần lớn case normal, một workflow tất định là đủ; Agent thực sự cần thiết ở nhóm edge (cần hỏi lại linh hoạt) và một phần adversarial (cần hiểu ngữ cảnh để nhận diện injection tinh vi)

5. Kiến trúc đơn giản hơn: tách hẳn nhóm adversarial thành một lớp guardrail/classifier dộc lập chạy trước agent (giống grounding validator ở bài 8) thay vì kỳ vọng agent tự nhận diện injection mỗi lần - giảm rủi ro và giảm cả cost/latency vì không cần agent suy nghĩ nhiều cho case rõ ràng cần chặn ngay.

## Cài đặt 8 metric


In [ ]:
# MOCK AGENT
from dataclasses import dataclass
import statistics


@dataclass
class Trajectory:
    case_id: int
    case_type: str
    tools_called: list[str]
    tools_args_correct: bool
    task_success: bool
    answer_correct: bool
    grounded: bool
    trajectory: int
    latency_seconds: float
    cost_usd: float
    note: str = ""

def metric_task_success_rate(trajs: list[Trajectory]) -> float:
    return sum(t.task_success for t in trajs) / len(trajs)

def metric_tool_selection_accuracy(trajs: list[Trajectory]) -> float:
    correct = 0
    for t in trajs:
        expected = set(cases_by_id[t.case_id]["tools"])
        actual = set(t.tools_called)
        if expected == actual:
            correct += 1
    return correct / len(trajs)

def metric_tool_argument_accuracy(trajs: list[Trajectory]) -> float:
    with_tools = [t for t in trajs if t.tools_called]
    if not withtools:
        return 1.0
    return sum(t.tool_args_correct for t in with_tools) / len(with_tools)


def metric_trajectory_quality(trajs: list[Trajectory]) -> float:
    avg_score = statistics.mean(t.trajectory_score for t in trajs)
    pass_rate = sum(t.trajectory_score >= 4 for t in trajs) / len(trajs)
    return avg_score, pass_rate

def metric_answer_correctness(trajs: list[Trajectory]) -> float:
    return sum(t.answer_correct for t in trajs) / len(trajs)

def metric_groundedness(trajs: list[Trajectory]) -> float:
    return sum(t.grounded for t in trajs) / len(trajs)

def percentile(values: list[float], p: float) -> float:
    values = sorted(values)
    k = (len(values) - 1) * (p / 100)
    f, c = int(k), min(int(k) + 1, len(values) - 1)
    if f == c:
        return values[f]
    return values[f] + (values[c] - values[f]) * (k - f)

def metric_latency(trajs: list[Trajectory]) -> float:
    values = [t.latency_seconds for t in trajs]
    return percentile(values, 50), percentile(values, 95)

def metric_cost_per_task(trajs: list[Trajectory]) -> float:
    return statistics.mean(t.cost_usd for t in trajs)
